# Decision-Tree Surrogate Explanations → CSV

This tutorial trains XGBoost and MLP models on the CoXAM `wine_quality` dataset,
fits a **decision-tree surrogate** (`DecisionTreeSurrogateMethod`) on each model's
predictions, generates per-instance explanations, and saves them to
`tutorials/generated_explanation/` in the standard project CSV formats.

**What a decision-tree surrogate is**

A surrogate is a simpler, interpretable model trained to mimic the black-box AI model.
Here, a shallow decision tree is fitted on the AI model's *predictions* (not ground-truth
labels). For each test instance the tree traces a decision path; features that appear
on that path receive an indicator value of `1.0`, all others `0.0`.

**Covered method**

| Registry key | Class | Description |
|---|---|---|
| `decision_tree` / `dt` / `rules` | `DecisionTreeSurrogateMethod` | Path-indicator explanations from a decision-tree surrogate |

**Output files** (saved to `tutorials/generated_explanation/`)

| File | Schema | Description |
|---|---|---|
| `dt_xgboost_wine_quality.csv` | Attribution | Per-instance path indicators — XGBoost surrogate |
| `dt_mlp_wine_quality.csv` | Attribution | Per-instance path indicators — MLP surrogate |
| `dt_xgboost_wine_quality_table.csv` | CoXAM surrogate | Global tree structure — XGBoost surrogate |
| `dt_mlp_wine_quality_table.csv` | CoXAM surrogate | Global tree structure — MLP surrogate |

## 1 · Setup

> **Important — run this cell first.**  
> `OMP_NUM_THREADS=1` and `MKL_NUM_THREADS=1` must be set **before** any import.
> On macOS, PyTorch's OpenMP runtime can conflict with the Jupyter kernel's fork model
> and cause a silent crash without these.

In [1]:
import os, io, contextlib
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

from pathlib import Path
import sys

repo_root = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "src").exists() and (candidate / "assets").exists()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from src.ai_models import MLPUnifiedModel, XGBoostUnifiedModel
from src.xai_adapter import DecisionTreeSurrogateMethod

OUTPUT_DIR = repo_root / "tutorials" / "generated_explanation"
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"repo_root:  {repo_root}")
print(f"output_dir: {OUTPUT_DIR}")

repo_root:  /Users/wangzhuoyulucas/Documents/GitHub/xaikit-test-api
output_dir: /Users/wangzhuoyulucas/Documents/GitHub/xaikit-test-api/tutorials/generated_explanation


## 2 · Load CoXAM Data

The CoXAM assets live in `assets/ai_dataset/coxam/`.  We use this source (400 instances)
rather than CoAX (120 instances) for more stable model training and surrogate fitting.

The CoXAM values table has columns `x0`–`x19` to accommodate different datasets.
For `wine_quality` only `x0`–`x5` carry data; the rest are `NaN`.  We filter to
non-`NaN` columns using the metadata row.

In [ ]:
ASSETS     = repo_root / "assets" / "ai_dataset" / "coxam"
DATASET_ID = "wine_quality"

values_df   = pd.read_csv(ASSETS / "values.csv")
metadata_df = pd.read_csv(ASSETS / "metadata.csv")

data = values_df[values_df["dataId"] == DATASET_ID].reset_index(drop=True)
meta = metadata_df[metadata_df["dataId"] == DATASET_ID].iloc[0]

# Keep only feature columns that have a name in the metadata (non-NaN)
all_xcols     = [c for c in data.columns if c.startswith("x") and c != "y"]
feature_cols  = [c for c in all_xcols if pd.notna(meta.get(c))]
feature_names = [str(meta[c]) for c in feature_cols]

X            = data[feature_cols].values.astype(float)
y            = data["y"].values.astype(int)
instance_ids = data["instanceId"].values

print(f"Dataset : {DATASET_ID}")
print(f"Shape   : {X.shape}")
print(f"Features: {list(zip(feature_cols, feature_names))}")
print(f"Classes : {dict(zip(*np.unique(y, return_counts=True)))}")
data[feature_cols + ["y", "instanceId"]].head()

Dataset : wine_quality
Shape   : (120, 5)
Features: [('x0', 'Vinegar Taint'), ('x1', 'SO2'), ('x2', 'pH'), ('x3', 'Sulphates'), ('x4', 'Alcohol')]
Classes : {np.int64(0): np.int64(64), np.int64(1): np.int64(56)}


,x0,x1,x2,x3,x4,y,instanceId
0,0.50,17.0,3.32,0.71,10.5,0,0
1,0.58,55.0,3.46,0.59,10.2,0,1
2,0.46,98.0,3.33,0.62,9.8,0,2
3,0.60,10.0,3.18,0.63,10.4,0,3
4,0.33,80.0,3.30,0.76,10.7,1,4


## 3 · Train AI Models

We train two built-in AI models on the same split:

- **XGBoost** (`XGBoostUnifiedModel`) — gradient-boosted trees, 50 boosting rounds.
- **MLP** (`MLPUnifiedModel`) — two-layer PyTorch perceptron, 300 epochs, batch size 32.

Both use `cognitive_agent='coxam'`.  Their **predictions** (not ground-truth labels)
are what the decision-tree surrogate will approximate.

> **Kernel-safe training**
> - XGBoost is trained **without a dev set** to prevent its C++ library from writing
>   verbose logs directly to the kernel's stdout pipe (which crashes the kernel).
> - MLP epoch output is captured with `contextlib.redirect_stdout` and suppressed.
> - In both cases accuracy is measured separately with `.evaluate()`.

In [10]:
X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(
    X, y, instance_ids, test_size=0.2, random_state=42
)
print(f"Train: {X_train.shape[0]} samples   Test: {X_test.shape[0]} samples")

Train: 96 samples   Test: 24 samples


In [11]:
xgb_model = XGBoostUnifiedModel(dataset_name=DATASET_ID, cognitive_agent="coxam")

# Train without X_dev: prevents XGBoost's C++ logger from flooding the kernel's stdout.
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    xgb_model.train(X_train, y_train)

xgb_acc = xgb_model.evaluate(X_test, y_test)
print(f"XGBoost trained ({buf.getvalue().count(chr(10))} log lines)")
print(f"XGBoost test accuracy: {xgb_acc:.3f}")

XGBoost trained (50 log lines)
XGBoost test accuracy: 0.958


In [12]:
mlp_model = MLPUnifiedModel(
    dataset_name=DATASET_ID,
    input_dim=X_train.shape[1],
    num_classes=2,
    cognitive_agent="coxam",
)

# Suppress per-epoch prints; smaller batches help convergence on this dataset.
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    mlp_model.train(X_train, y_train, epochs=300, batch_size=32)

mlp_acc = mlp_model.evaluate(X_test, y_test)
print(f"MLP trained ({buf.getvalue().count('Epoch')} epochs, output suppressed)")
print(f"MLP test accuracy: {mlp_acc:.3f}")

MLP trained (300 epochs, output suppressed)
MLP test accuracy: 0.750


## 4 · Generate Model Predictions for Surrogate Fitting

A surrogate approximates the **AI model's behaviour**, not the ground-truth labels.
We call `.predict()` on the training set and take `argmax` to convert probability
arrays to class indices.  These become the `y` that the decision-tree surrogate trains on.

`model.predict(X)` returns an `(n_samples, n_classes)` probability array.

In [13]:
y_xgb = np.argmax(xgb_model.predict(X_train), axis=1)
y_mlp = np.argmax(mlp_model.predict(X_train), axis=1)

print(f"XGBoost train label dist: {dict(zip(*np.unique(y_xgb, return_counts=True)))}")
print(f"MLP     train label dist: {dict(zip(*np.unique(y_mlp, return_counts=True)))}")

XGBoost train label dist: {np.int64(0): np.int64(54), np.int64(1): np.int64(42)}
MLP     train label dist: {np.int64(0): np.int64(50), np.int64(1): np.int64(46)}


## 5 · Fit Decision-Tree Surrogate

`dt.fit(X_train, y_model)` fits a shallow decision tree (`depth=3`) to the model
predictions.  Internally it calls `generate_decision_tree_table()` which stores the
full tree structure as JSON and computes **fidelity** — the fraction of training
instances where the surrogate agrees with the AI model.

Key constructor parameters:
- `app_id`        — written to the `dataId` column in output CSVs
- `model_name`    — written to the `modelName` column in output CSVs
- `depth`         — maximum depth of the surrogate tree
- `feature_names` — human-readable names stored in the metadata table

In [14]:
dt_xgb = DecisionTreeSurrogateMethod(
    app_id=DATASET_ID,
    model_name="xgboost",
    depth=3,
    feature_names=feature_names,
)
dt_xgb.fit(X_train, y_xgb)

print(f"DT surrogate (XGBoost)  fidelity : {dt_xgb.fidelity:.3f}")
print(f"                        nodes    : {len(dt_xgb.tree_structure)}")

DT surrogate (XGBoost)  fidelity : 0.917
                        nodes    : 13


In [15]:
dt_mlp = DecisionTreeSurrogateMethod(
    app_id=DATASET_ID,
    model_name="mlp",
    depth=3,
    feature_names=feature_names,
)
dt_mlp.fit(X_train, y_mlp)

print(f"DT surrogate (MLP)      fidelity : {dt_mlp.fidelity:.3f}")
print(f"                        nodes    : {len(dt_mlp.tree_structure)}")

DT surrogate (MLP)      fidelity : 0.958
                        nodes    : 13


## 6 · Compare Fidelity

Fidelity measures how well the surrogate reproduces the AI model's training-set
predictions.  Higher fidelity means the tree is a better proxy.  Two models can differ
in fidelity at the same depth — tree-structured models like XGBoost are generally easier
to approximate with a decision-tree surrogate.

In [16]:
pd.DataFrame({
    "surrogate" : ["DT depth=3 → XGBoost", "DT depth=3 → MLP"],
    "fidelity"  : [dt_xgb.fidelity, dt_mlp.fidelity],
    "nodes"     : [len(dt_xgb.tree_structure), len(dt_mlp.tree_structure)],
    "model_acc" : [xgb_acc, mlp_acc],
})

,surrogate,fidelity,nodes,model_acc
0,DT depth=3 → XGBoost,0.916667,13,0.958333
1,DT depth=3 → MLP,0.958333,13,0.750000


## 7 · Explain Test Instances

`dt.explain(X_test)` runs each test instance through the fitted surrogate tree.
Features on its decision path receive `1.0`; all others receive `0.0`.

The result is an `XAIAdapterResult` with:
- `result.values`      — `(n_instances, n_features)` path-indicator matrix
- `result.base_values` — `(n_instances,)` zeros (no base offset for tree surrogates)
- `result.method`      — `'decision_tree'`
- `result.metadata`    — fidelity, app_id, model_name, per-instance decision paths

In [17]:
preds_xgb_test = np.argmax(xgb_model.predict(X_test), axis=1)
preds_mlp_test = np.argmax(mlp_model.predict(X_test), axis=1)

result_dt_xgb = dt_xgb.explain(X_test)
result_dt_mlp = dt_mlp.explain(X_test)

print(f"DT/XGBoost  shape={result_dt_xgb.values.shape}  method='{result_dt_xgb.method}'")
print(f"DT/MLP      shape={result_dt_mlp.values.shape}  method='{result_dt_mlp.method}'")

DT/XGBoost  shape=(24, 5)  method='decision_tree'
DT/MLP      shape=(24, 5)  method='decision_tree'


In [18]:
# Preview: standard explanation DataFrame
# Schema: dataId, modelName, expMethod, instanceId, pred, i_max, a0_i…aN_i, intercept
df_dt_xgb = result_dt_xgb.to_explanation_df(ids_test, preds_xgb_test, DATASET_ID, "xgboost")
print(f"Shape: {df_dt_xgb.shape}  columns: {df_dt_xgb.columns.tolist()}")
df_dt_xgb.head(3)

Shape: (24, 12)  columns: ['dataId', 'modelName', 'expMethod', 'instanceId', 'pred', 'i_max', 'a0_i', 'a1_i', 'a2_i', 'a3_i', 'a4_i', 'intercept']


,dataId,modelName,expMethod,instanceId,pred,i_max,a0_i,a1_i,a2_i,a3_i,a4_i,intercept
0,wine_quality,xgboost,decision_tree,44,0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
1,wine_quality,xgboost,decision_tree,47,1,1.0,1.0,0.0,0.0,1.0,1.0,0.0
2,wine_quality,xgboost,decision_tree,4,1,1.0,1.0,0.0,0.0,1.0,1.0,0.0


## 8 · Save Per-Instance Explanation CSV

`result.save_csv(path, instance_ids, predictions, dataset_id, model_name)` writes the
standard attribution schema to disk (creates parent dirs automatically).

Schema: `dataId, modelName, expMethod, instanceId, pred, i_max, a0_i…aN_i, intercept`
- `aN_i`     — path-indicator for feature N, normalized by `i_max` (0.0 or 1.0)
- `i_max`    — max absolute indicator (= 1.0 for binary path indicators)
- `intercept`— base value (= 0.0 for decision-tree surrogates)

In [19]:
xgb_attr_path = OUTPUT_DIR / f"dt_xgboost_{DATASET_ID}.csv"
mlp_attr_path = OUTPUT_DIR / f"dt_mlp_{DATASET_ID}.csv"

result_dt_xgb.save_csv(xgb_attr_path, ids_test, preds_xgb_test, DATASET_ID, "xgboost")
result_dt_mlp.save_csv(mlp_attr_path, ids_test, preds_mlp_test, DATASET_ID, "mlp")

print(f"Saved → {xgb_attr_path.relative_to(repo_root)}")
print(f"Saved → {mlp_attr_path.relative_to(repo_root)}")

Saved → tutorials/generated_explanation/dt_xgboost_wine_quality.csv
Saved → tutorials/generated_explanation/dt_mlp_wine_quality.csv


## 9 · Save Surrogate Model Tables

`dt.to_explanation_table()` returns the CoXAM-style surrogate table — one row per
fitted model — containing the full tree structure as JSON and the fidelity score.
This matches the schema of `assets/explanations/coxam/decision_tree.csv`.

`dt.to_metadata_table()` returns feature-level metadata (names, min/max bounds).

In [20]:
xgb_table_path = OUTPUT_DIR / f"dt_xgboost_{DATASET_ID}_table.csv"
mlp_table_path = OUTPUT_DIR / f"dt_mlp_{DATASET_ID}_table.csv"

dt_xgb.to_explanation_table().to_csv(xgb_table_path, index=False)
dt_xgb.to_metadata_table().to_csv(
    OUTPUT_DIR / f"dt_xgboost_{DATASET_ID}_metadata.csv", index=False
)
dt_mlp.to_explanation_table().to_csv(mlp_table_path, index=False)
dt_mlp.to_metadata_table().to_csv(
    OUTPUT_DIR / f"dt_mlp_{DATASET_ID}_metadata.csv", index=False
)

print(f"Saved → {xgb_table_path.relative_to(repo_root)}")
print(f"Saved → {mlp_table_path.relative_to(repo_root)}")
print(f"\nSurrogate table columns: {dt_xgb.to_explanation_table().columns.tolist()}")

Saved → tutorials/generated_explanation/dt_xgboost_wine_quality_table.csv
Saved → tutorials/generated_explanation/dt_mlp_wine_quality_table.csv

Surrogate table columns: ['dataId', 'model', 'depth', 'fidelity', 'tree_structure', 'class_labels']


## 10 · Reload and Verify

In [ ]:
attr_df = pd.read_csv(OUTPUT_DIR / f"dt_xgboost_{DATASET_ID}.csv")
print("Attribution CSV columns:", attr_df.columns.tolist())
print(f"Shape: {attr_df.shape}")
attr_df.head(3)

In [ ]:
table_df = pd.read_csv(OUTPUT_DIR / f"dt_xgboost_{DATASET_ID}_table.csv")
print("Surrogate table columns:", table_df.columns.tolist())
print(f"Fidelity: {table_df['fidelity'].iloc[0]:.3f}  Depth: {table_df['depth'].iloc[0]}")

---
## Summary

| Step | What happened |
|---|---|
| Data | Loaded `wine_quality` (400 rows, 6 features) from `assets/ai_dataset/coxam/values.csv` |
| Models | XGBoost (50 rounds) and MLP (300 epochs, batch=32) via built-in model classes |
| Predictions | `y_model = np.argmax(model.predict(X_train), axis=1)` |
| Fit surrogate | `DecisionTreeSurrogateMethod.fit(X_train, y_model)` — depth 3 |
| Explain | `dt.explain(X_test)` → path-indicator matrix (1 = feature on decision path) |
| Attribution CSVs | `dt_{xgboost,mlp}_wine_quality.csv` |
| Surrogate tables | `dt_{xgboost,mlp}_wine_quality_table.csv` (CoXAM format, tree JSON) |

**API reference**

```python
# 1. Create and fit
dt = DecisionTreeSurrogateMethod(
    app_id="...", model_name="...", depth=3, feature_names=[...]
)
dt.fit(X_train, np.argmax(model.predict(X_train), axis=1))

# 2. Explain test instances
result = dt.explain(X_test)

# 3. Save per-instance explanation CSV
result.save_csv(path, instance_ids, predictions, dataset_id, model_name)

# 4. Save global surrogate structure tables
dt.to_explanation_table().to_csv(table_path, index=False)
dt.to_metadata_table().to_csv(metadata_path, index=False)
```